In [3]:
!pip install optuna

  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 24.1 MB/s eta 0:00:00
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)

   ----- ---------------------------------- 1/8 [PyYAML]
   ---------- ----------------------------- 2/8 [Mako]
   --------------- ------------------------ 3/8 [greenlet]
   ------------------------- -------------- 5/8 [sqlalchemy]
   ------------------------- -------------- 5/8 [sqlalchemy]
   ------------------------- -------------- 5/8 [sqlalchemy]
   ------------------------- -------------- 5/8 [sqlalchemy]
   ------------------------- -------------- 5/8 [sqlalchemy]
   ------------------------- -------------- 5/8 [sqlalchemy]
   ------------------------- -------------- 5/8 [sqlalchemy]
   ------------------------- -------------- 5/8 [sqlalchemy]
   ------------------------- -------------- 5/8 [sqlalchemy]
   -------------------


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# xgboost_optuna_feature_selection.py
# 需要安裝: xgboost, optuna, scikit-learn, pandas, numpy
# pip install xgboost optuna scikit-learn pandas numpy

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import roc_auc_score, make_scorer
from xgboost import XGBClassifier
import optuna
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

def optuna_objective(trial, X, y, n_splits=5, random_state=RANDOM_STATE):
    
    # 計算 pos/neg 比例（資料驅動）
    neg = (y == 0).sum()
    pos = (y == 1).sum()
    ratio = neg / pos

    # 超參數空間（可依需求擴展）
    param = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_loguniform("learning_rate", 1e-3, 0.3),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "reg_alpha": trial.suggest_loguniform("reg_alpha", 1e-8, 10.0),
        "reg_lambda": trial.suggest_loguniform("reg_lambda", 1e-8, 10.0),
        # ⭐⭐⭐ 加入 scale_pos_weight（關鍵）
        # 通常 1 ~ neg/pos，但不需精準，讓 optuna 搜
        "scale_pos_weight": trial.suggest_float(
            "scale_pos_weight", 
            1.0, 
            ratio
        ),
        "use_label_encoder": False,
        "eval_metric": "auc",
        "random_state": random_state,
        "verbosity": 0,
    }

    # Stratified 5-fold CV
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    model = XGBClassifier(**param)

    # 以 AUC 做為評分
    aucs = cross_val_score(model, X, y, cv=skf, scoring="roc_auc", n_jobs=-1)
    return float(np.mean(aucs))


def find_best_params_with_optuna(X, y, n_trials=50, n_splits=5):
    func = lambda trial: optuna_objective(trial, X, y, n_splits=n_splits)
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(func, n_trials=n_trials, show_progress_bar=True)
    print("Best trial value (CV AUC):", study.best_value)
    print("Best params:", study.best_trial.params)
    best_params = study.best_trial.params

    # add fixed params for XGBClassifier usage
    best_params.update({
        "use_label_encoder": False,
        "eval_metric": "auc",
        "random_state": RANDOM_STATE,
        "verbosity": 0
    })
    return best_params, study


def feature_selection_by_importance(X, y, best_params, n_splits=5, thresholds=None):
    """
    使用 SelectFromModel 基於 feature_importances_ 的 threshold 選特徵
    thresholds: list of relative thresholds (如 [0.0, 0.01, 0.05, 0.1, ...]) 或 None -> 自動以 percentiles 產生
    回傳: dict 包含最佳特徵清單、最佳 CV AUC、所有嘗試結果 DataFrame
    """
    if thresholds is None:
        # 產生一組百分位 threshold（0% 到 99%）
        percentiles = np.concatenate([np.linspace(0, 90, 10), np.linspace(90, 99, 10)])
        thresholds = percentiles / 100.0

    base_model = XGBClassifier(**best_params)
    # 先在全部資料上 fit 得到 feature_importances_
    base_model.fit(X, y)
    importances = base_model.feature_importances_
    feat_names = np.array(X.columns)

    results = []
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    # Convert importances to percentile-based thresholds:
    sorted_imp = np.sort(importances)
    for t in thresholds:
        if t <= 0:
            thresh_value = 0.0
        else:
            # threshold value as percentile of importances
            thresh_value = np.percentile(importances, t * 100)
        sel = SelectFromModel(base_model, threshold=thresh_value, prefit=True)
        try:
            selected_mask = sel.get_support()
        except Exception:
            # fallback: if seletor fails, skip
            continue
        n_selected = selected_mask.sum()
        if n_selected == 0:
            # skip empty selection
            continue
        X_sel = X.loc[:, selected_mask]

        # Evaluate selected features with CV (same model and params)
        model = XGBClassifier(**best_params)
        aucs = cross_val_score(model, X_sel, y, cv=skf, scoring="roc_auc", n_jobs=-1)
        mean_auc = float(np.mean(aucs))
        results.append({
            "threshold_percentile": t,
            "thresh_value": thresh_value,
            "n_selected": int(n_selected),
            "cv_auc": mean_auc
        })

    results_df = pd.DataFrame(results).sort_values("cv_auc", ascending=False).reset_index(drop=True)
    if results_df.shape[0] == 0:
        raise RuntimeError("No feature subset selected — check thresholds or importances.")
    best_row = results_df.iloc[0]
    # get selected features for best threshold
    best_thresh_val = best_row["thresh_value"]
    sel_best = SelectFromModel(base_model, threshold=best_thresh_val, prefit=True)
    best_mask = sel_best.get_support()
    selected_features = list(feat_names[best_mask])

    return {
        "selected_features": selected_features,
        "best_cv_auc": float(best_row["cv_auc"]),
        "results_df": results_df
    }


def train_final_model(X, y, selected_features, best_params):
    """
    在全部資料上用選到的特徵訓練最終模型 (可以後續保存 model)
    回傳訓練好的 model
    """
    X_sel = X[selected_features].copy()
    model = XGBClassifier(**best_params)
    model.fit(X_sel, y)
    return model



c:\Users\user\Documents\ExpertBook\2025\用戶資料集\TrainData\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("forTrain_final_merged_noSYS.csv", sep="^")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85118 entries, 0 to 85117
Columns: 202 entries, CUST_NO to 使用狀態_編碼
dtypes: float64(184), int64(16), object(2)
memory usage: 131.2+ MB


In [4]:
# Check object columns in sfss_final_merged
obj_cols = df.select_dtypes(include=['object']).columns.tolist()
print('Object columns count:', len(obj_cols))
print('Object columns:', obj_cols)

Object columns count: 2
Object columns: ['工單日期', '使用狀態']


In [5]:
# 指定排除欄位, 注意類似 工單日期屬於 object dtype 要刪掉，避免影響後續分析
exclude_cols = ['使用狀態','工單日期']
df = df.drop(columns=exclude_cols)

In [6]:
# 讀取並準備資料
# 假設你已經有 X, y：
# X: pandas DataFrame (n_samples, n_features)
# y: pandas Series 或 1D array (n_samples,)
# 這裡給出一個模板：你應該把下面兩行替換成實際資料讀取與前處理
# -----------------------------------------
# Example load (replace with real data):
# df = pd.read_csv("churn_data.csv")
# y = df["churn"]
# X = df.drop(columns=["churn", "customer_id"])
# -----------------------------------------

y = df["使用狀態_編碼"]
X = df.drop(columns=["使用狀態_編碼", "CUST_NO"])

# ---- 若你要快速測試，可啟用以下 synthetic 範例 (小資料) ----
# from sklearn.datasets import make_classification
# X_np, y_np = make_classification(n_samples=1000, n_features=40, n_informative=8,
#                                  weights=[0.8, 0.2], random_state=RANDOM_STATE)
# X = pd.DataFrame(X_np, columns=[f"f{i}" for i in range(X_np.shape[1])])
# y = pd.Series(y_np)

# ----------------------------------------------------------------
# 下面開始真正的 pipeline（假設 X, y 已存在）
# ----------------------------------------------------------------
# Replace the following two lines with your actual X and y:
# X = ...
# y = ...
# ----------------------------------------------------------------

# 如果你把這支當 script 執行，請先在上面填入 X, y，或把 raise_if_demo 改為 True 來啟動 synthetic 範例。
# 下面範例呼叫流程（請確保 X, y 已被定義）：
try:
    X  # check defined
    y
except Exception:
    print("請先在 script 中定義 X (DataFrame) 與 y (Series/array)。或啟用 synthetic 範例以測試。")
    raise SystemExit(1)

# 1) Optuna 搜尋最佳參數
best_params, study = find_best_params_with_optuna(X, y, n_trials=50, n_splits=5)

# 2) 基於 feature importances 做特徵選擇（threshold 以 percentiles 嘗試）
fs_result = feature_selection_by_importance(X, y, best_params, n_splits=5, thresholds=None)
print("Best CV AUC after feature selection:", fs_result["best_cv_auc"])
print("Selected features ({}):".format(len(fs_result["selected_features"])))
print(fs_result["selected_features"])

# 3) 訓練最終模型
final_model = train_final_model(X, y, fs_result["selected_features"], best_params)
print("Final model trained on selected features.")


[I 2025-12-23 20:14:46,368] A new study created in memory with name: no-name-07f5aa26-c0ca-40e5-9401-d73b30e2e629
Best trial: 0. Best value: 0.725699:   2%|▏         | 1/50 [00:38<31:47, 38.92s/it]

[I 2025-12-23 20:15:25,284] Trial 0 finished with value: 0.7256987399375001 and parameters: {'n_estimators': 406, 'max_depth': 12, 'learning_rate': 0.06504856968981275, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.4936111842654619, 'gamma': 0.7799726016810132, 'min_child_weight': 2, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.002570603566117598, 'scale_pos_weight': 10.90423832991576}. Best is trial 0 with value: 0.7256987399375001.


Best trial: 1. Best value: 0.745541:   4%|▍         | 2/50 [00:52<19:02, 23.81s/it]

[I 2025-12-23 20:15:38,510] Trial 1 finished with value: 0.7455411010359783 and parameters: {'n_estimators': 69, 'max_depth': 12, 'learning_rate': 0.11536162338241392, 'subsample': 0.6061695553391381, 'colsample_bytree': 0.5090949803242604, 'gamma': 0.9170225492671691, 'min_child_weight': 7, 'reg_alpha': 0.00052821153945323, 'reg_lambda': 7.71800699380605e-05, 'scale_pos_weight': 5.07359768417509}. Best is trial 1 with value: 0.7455411010359783.


Best trial: 1. Best value: 0.745541:   6%|▌         | 3/50 [01:28<23:05, 29.47s/it]

...

[I 2025-12-23 20:40:17,401] Trial 47 finished with value: 0.7486213327244391 and parameters: {'n_estimators': 449, 'max_depth': 9, 'learning_rate': 0.026538085566854008, 'subsample': 0.9375011965514362, 'colsample_bytree': 0.8938344982395143, 'gamma': 2.5460442988182117, 'min_child_weight': 14, 'reg_alpha': 7.932789538357011e-07, 'reg_lambda': 1.5421161821076286, 'scale_pos_weight': 7.283246804160161}. Best is trial 31 with value: 0.7519296380236101.


Best trial: 31. Best value: 0.75193:  98%|█████████▊| 49/50 [26:04<00:27, 27.50s/it]

[I 2025-12-23 20:40:50,905] Trial 48 finished with value: 0.7469245797229602 and parameters: {'n_estimators': 642, 'max_depth': 6, 'learning_rate': 0.03949507438235698, 'subsample': 0.7508624958812509, 'colsample_bytree': 0.8519169522464884, 'gamma': 2.269528935523691, 'min_child_weight': 17, 'reg_alpha': 3.9662770231168987e-05, 'reg_lambda': 2.398951982705002e-05, 'scale_pos_weight': 11.471139503302318}. Best is trial 31 with value: 0.7519296380236101.


Best trial: 31. Best value: 0.75193: 100%|██████████| 50/50 [26:51<00:00, 32.22s/it]


[I 2025-12-23 20:41:37,490] Trial 49 finished with value: 0.7509670421760142 and parameters: {'n_estimators': 732, 'max_depth': 8, 'learning_rate': 0.011733663843862997, 'subsample': 0.8293341656731361, 'colsample_bytree': 0.9733768941036235, 'gamma': 0.4721934023323665, 'min_child_weight': 19, 'reg_alpha': 1.0521249699838866e-07, 'reg_lambda': 4.322321806186226e-08, 'scale_pos_weight': 7.995541515945124}. Best is trial 31 with value: 0.7519296380236101.
Best trial value (CV AUC): 0.7519296380236101
Best params: {'n_estimators': 413, 'max_depth': 6, 'learning_rate': 0.02148122476467495, 'subsample': 0.7293727282669078, 'colsample_bytree': 0.8833439518299182, 'gamma': 2.056973872905811, 'min_child_weight': 19, 'reg_alpha': 1.3904716634190133e-06, 'reg_lambda': 1.906552145742653e-05, 'scale_pos_weight': 7.752118844979001}
Best CV AUC after feature selection: 0.7531463640441749
Selected features (60):
['平均繳款延遲日數', 'paytype_1', 'paytype_6', 'paytype_15', 'paytype_99', 'paymethod_信用卡扣款', 'p

In [5]:
print(final_model)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9331716140575979, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='auc', feature_types=None, feature_weights=None,
              gamma=0.6142722657333379, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.059545500099541224,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=12, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=369, n_jobs=None,
              num_parallel_tree=None, ...)
